# Hip Implant — Contralateral Mirror Reconstruction

Build a custom acetabular implant for a **unilateral hip defect** by mirroring the patient's healthy contralateral hip and computing the difference. No training, no GPU required — runs in seconds on Colab CPU.

**Inputs you need:** two STL files exported from the patient's CT segmentation (e.g. via TotalSegmentator + 3D Slicer):
1. The **defective** hip bone (left or right, the one with bone loss)
2. The **healthy contralateral** hip bone

**Output:** `implant.stl` — the geometry that fills the missing region, ready to send to your slicer.

**Why mirror, not AI?** When the healthy contralateral side exists, it's the gold standard reconstruction reference because it's the patient's own anatomy. AI shape-completion is reserved for cases where mirroring isn't possible (bilateral defects, etc.) — see `train_colab.ipynb`.

---

## Run the cell below — it does everything

In [ ]:
# =====================================================================
# ALL-IN-ONE: clone repo if missing -> upload both hips -> reconstruct
#             -> 3D preview -> download implant.stl
# =====================================================================
import sys, os, subprocess

REPO = '/content/Claude-hip-try1'
BRANCH = 'claude/hip-bone-reconstruction-bGrN1'
if not os.path.isdir(REPO):
    subprocess.run(
        ['git', 'clone', '-b', BRANCH,
         'https://github.com/bhaskarsdose/Claude-hip-try1.git', REPO],
        check=True,
    )
subprocess.run(['git', '-C', REPO, 'pull', '-q'], check=False)

# Install runtime deps (idempotent, ~30s on first run, instant after)
subprocess.run(
    ['pip', 'install', '-q',
     'trimesh>=4.0', 'numpy>=1.24', 'scipy>=1.11',
     'scikit-image>=0.22', 'plotly>=5.20'],
    check=True,
)

SRC = f'{REPO}/src'
if SRC not in sys.path:
    sys.path.insert(0, SRC)

import importlib, hip_recon.mirror
importlib.reload(hip_recon.mirror)
from hip_recon.mirror import mirror_reconstruct
from google.colab import files
import numpy as np
import plotly.graph_objects as go

# 1) Upload both STLs
print('1) Choose the DEFECTIVE hip STL...')
defective_path = next(iter(files.upload()))
print('   defective:', defective_path)
print('\n2) Choose the HEALTHY contralateral hip STL...')
healthy_path = next(iter(files.upload()))
print('   healthy:  ', healthy_path)

# 2) Reconstruct
mres = mirror_reconstruct(
    defective_path, healthy_path,
    mirror_axis=0,        # try 1 or 2 if implant is on wrong side
    dilate_input=0,       # 0 = snug fit, 1 = small cement gap, 3 = safe margin
    cleanup_iters=3,      # 3 severs thin 'fingers'; 4-5 = more aggressive
    smooth_iters=15,
    keep_largest=True,
)
print(f'\nImplant: {len(mres.implant_mesh.vertices)} verts / '
      f'{len(mres.implant_mesh.faces)} faces')

# 3) Inline 3D preview
def trace(m, c, n, op=0.85):
    if len(m.vertices) == 0: return None
    v, f = np.asarray(m.vertices), np.asarray(m.faces)
    return go.Mesh3d(x=v[:,0], y=v[:,1], z=v[:,2],
                     i=f[:,0], j=f[:,1], k=f[:,2],
                     color=c, opacity=op, name=n, showlegend=True)

fig = go.Figure(data=[t for t in [
    trace(mres.defective_mesh, '#b8c0cc', 'Defective bone', 0.55),
    trace(mres.implant_mesh,   '#ff9b3d', 'Implant',         0.95),
] if t])
fig.update_layout(scene=dict(aspectmode='data'), height=720,
                  title='Hip implant — mirror reconstruction')
fig.show()

# 4) Save + download
os.makedirs('outputs', exist_ok=True)
out = 'outputs/implant.stl'
mres.implant_mesh.export(out)
print(f'\nSaved {out}')
files.download(out)

## Tuning knobs (only if the default output isn't right)

Edit these in the cell above and re-run:

| Parameter | Effect |
|---|---|
| `mirror_axis=0` | Axis to flip across. Default 0 (X) is correct for TotalSegmentator's RAS orientation. Try 1 or 2 if implant ends up on the wrong side of the body. |
| `dilate_input=0` | Voxels of clearance between implant and existing bone. 0 = snug print fit, 1 = ~0.8mm gap for cement, 3 = ~2mm safety margin. |
| `cleanup_iters=3` | Morphological opening passes. Higher value severs more thin protrusions; 5+ keeps only the central reconstruction. |
| `smooth_iters=15` | Taubin surface smoothing iterations. 0 = raw marching-cubes faceting, 30+ = very smooth. |

## Adding the femoral-head socket (optional)

If you need an acetabular cup carved into the implant (so it articulates with a femoral head), add this cell after the all-in-one cell:

```python
from hip_recon.mirror import detect_acetabulum, add_femoral_socket
cup_centre, cup_radius = detect_acetabulum(mres.mirrored_mesh)
implant_with_cup = add_femoral_socket(mres.implant_mesh, cup_centre, cup_radius, keep_largest=True)
implant_with_cup.export('outputs/implant_with_cup.stl')
files.download('outputs/implant_with_cup.stl')
```

Override `cup_centre = np.array([X, Y, Z])` and `cup_radius = 23.0` manually if the auto-detect picks the wrong location.